# Airbnb Berlin: what drives nightly price?

**Pet project / Exploratory Data Analysis**  
Данные: Inside Airbnb, Berlin listings (2025).

Проект превращает курсовое исследование в воспроизводимый EDA: очистка, визуализации и проверка факторов, связанных с ценой за ночь.

## 1. Setup and data loading

Ноутбук использует точный срез [Inside Airbnb — Berlin](https://insideairbnb.com/get-the-data/) от **23 сентября 2025 года**. Если файла нет рядом с ноутбуком, код скачает его автоматически.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

DATA_PATH = Path('listings.csv.gz')
DATA_URL = ('https://data.insideairbnb.com/germany/be/berlin/'
            '2025-09-23/data/listings.csv.gz')
if not DATA_PATH.exists():
    from urllib.request import urlretrieve
    print('Downloading the 23 Sep 2025 Berlin snapshot…')
    urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(f'Raw rows: {df.shape[0]:,}; columns: {df.shape[1]:,}')
df.head()

## 2. Cleaning

Цена приводится к числу; экстремальные значения выше 99-го перцентиля исключаются, чтобы они не искажали сравнения.

In [ ]:
df = df.copy()
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)
p99 = df['price'].quantile(0.99)
df_clean = df.loc[df['price'].le(p99)].copy()

cols = [
    'price', 'room_type', 'neighbourhood_group_cleansed', 'accommodates',
    'bedrooms', 'bathrooms', 'beds', 'minimum_nights',
    'number_of_reviews', 'review_scores_rating', 'host_is_superhost',
    'availability_365'
]
df_clean = df_clean[cols]
print(f'99th percentile: €{p99:,.0f}')
print(f'Rows after outlier treatment: {len(df_clean):,}')
df_clean['price'].describe()

## 3. Price distribution and housing type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_clean['price'], bins=45, color='#3856A6', edgecolor='white')
axes[0].set(title='Nightly price distribution', xlabel='Price, €', ylabel='Listings')

room_order = ['Entire home/apt', 'Private room', 'Shared room', 'Hotel room']
sns.boxplot(data=df_clean, x='room_type', y='price', order=room_order, ax=axes[1], showfliers=False)
axes[1].set(title='Price by room type', xlabel='', ylabel='Price, €')
axes[1].tick_params(axis='x', rotation=18)
fig.tight_layout()
fig.savefig(FIG_DIR / '01_price_and_room_type.png', dpi=180, bbox_inches='tight')
plt.show()

df_clean.groupby('room_type')['price'].agg(['count', 'mean', 'median']).sort_values('mean', ascending=False).round(1)

## 4. Location and capacity

In [ ]:
by_neigh = (df_clean.groupby('neighbourhood_group_cleansed')['price']
              .agg(['count', 'mean', 'median'])
              .sort_values('mean', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
by_neigh['mean'].sort_values().plot.barh(ax=axes[0], color='#2F7D6D')
axes[0].set(title='Mean nightly price by district', xlabel='Price, €', ylabel='')

sns.scatterplot(data=df_clean, x='accommodates', y='price', alpha=.22, s=18, ax=axes[1], color='#3856A6')
axes[1].set(title='Capacity is positively associated with price', xlabel='Accommodates', ylabel='Price, €')
fig.tight_layout()
fig.savefig(FIG_DIR / '02_location_and_capacity.png', dpi=180, bbox_inches='tight')
plt.show()

by_neigh.round(1)

## 5. What matters less?

Корреляции не доказывают причинность, но помогают увидеть, какие признаки стоит брать в следующую модель.

In [ ]:
numeric_cols = ['price', 'accommodates', 'bedrooms', 'bathrooms', 'beds',
                'minimum_nights', 'number_of_reviews', 'review_scores_rating',
                'availability_365']
corr = df_clean[numeric_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.scatterplot(data=df_clean, x='minimum_nights', y='price', alpha=.2, s=16, ax=axes[0], color='#B7554A')
axes[0].set(xlim=(0, 60), title='Longer minimum stays tend to cost less per night',
            xlabel='Minimum nights', ylabel='Price, €')

sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1, square=True, ax=axes[1])
axes[1].set_title('Correlation matrix')
fig.tight_layout()
fig.savefig(FIG_DIR / '03_min_nights_and_correlations.png', dpi=180, bbox_inches='tight')
plt.show()

corr['price'].sort_values(ascending=False).round(2)

## 6. Findings and next step

- **Key levers:** room type, district, capacity / bedrooms and minimum stay.
- **Weak direct signals:** rating, review count and availability.
- **Next step:** split train/test data and fit a regularized regression or gradient boosting model; add listing text, coordinates and time effects.